# 📊 Notebook 08: Baseline Model Comparison
## Explainable GNNs for ASD Classification — Comparative Study

**Goal:** Justify why GAT outperforms simpler approaches by comparing against:
1. **SVM** — Classical ML on flattened connectivity matrices (no graph structure)
2. **MLP** — Deep learning on flattened node features (no message passing)
3. **GCN** — Graph Convolutional Network (no attention mechanism)
4. **Improved GAT** — Our method from Notebook 07

| Model | Captures Graph Structure | Learnable Attention | Explainability |
|---|---|---|---|
| SVM | ❌ (flattened FC matrix) | ❌ | ❌ |
| MLP | ❌ (flattened features) | ❌ | Partial |
| GCN | ✅ | ❌ (fixed norm) | Partial |
| **ImprovedGAT** | ✅ | ✅ | ✅ GNNExplainer + IG |

> ⚡ **Runtime:** ~20–30 minutes total on CPU. GPU speeds up GCN/GAT significantly.

In [1]:
# ─── CELL 1: Install ──────────────────────────────────────────────────────────
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric', '-q'], check=True)
import torch
cuda_tag = 'cu' + torch.version.cuda.replace('.','') if torch.cuda.is_available() else 'cpu'
pyg_url  = f'https://data.pyg.org/whl/torch-{torch.__version__.split("+")[0]}+{cuda_tag}.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv',
    '-f', pyg_url], check=False)

print(f"✅ Setup complete! PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

✅ Setup complete! PyTorch 2.11.0+cu128 | CUDA: True


In [2]:
# ─── CELL 2: All Imports ──────────────────────────────────────────────────────
import os
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d
from torch_geometric.loader import DataLoader
from torch_geometric.data import Dataset
from torch_geometric.nn import (GATConv, GCNConv,
                                  global_mean_pool, global_max_pool)

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, make_scorer
)

from google.colab import drive

plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.titlesize': 12})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Imports done | Device: {DEVICE}")

✅ Imports done | Device: cuda


In [3]:
# ─── CELL 3: Mount Drive & Paths ──────────────────────────────────────────────
drive.mount('/content/drive')

BASE_DIR     = '/content/drive/MyDrive/ASD_GNN_Research2'
GRAPHS_DIR   = os.path.join(BASE_DIR, 'graphs')
IMPROVED_DIR = os.path.join(BASE_DIR, 'models', 'GAT')
METRICS_DIR  = os.path.join(BASE_DIR, 'results', 'metrics')
FIGURES_DIR  = os.path.join(BASE_DIR, 'results', 'figures')
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
print("✅ Drive mounted")

Mounted at /content/drive
✅ Drive mounted


In [4]:
# ─── CELL 4: Model Definitions ────────────────────────────────────────────────

# ── Dataset ──────────────────────────────────────────────────────────────────
class ABIDEGraphDataset(Dataset):
    def __init__(self, folder_path):
        super().__init__()
        self.folder_path = folder_path
        self.file_names  = sorted([f for f in os.listdir(folder_path) if f.endswith('.pt')])
    def len(self): return len(self.file_names)
    def get(self, idx):
        return torch.load(os.path.join(self.folder_path, self.file_names[idx]),
                           weights_only=False)


# ── 1. MLP Baseline ──────────────────────────────────────────────────────────
class MLPClassifier(torch.nn.Module):
    """
    MLP on mean-pooled node features [1, 6].
    No graph structure — treats each brain as a feature vector.
    """
    def __init__(self, in_features=6, hidden=128, num_classes=2, dropout=0.4):
        super().__init__()
        self.net = torch.nn.Sequential(
            Linear(in_features, hidden),
            BatchNorm1d(hidden), torch.nn.ELU(),
            torch.nn.Dropout(dropout),
            Linear(hidden, hidden // 2),
            BatchNorm1d(hidden // 2), torch.nn.ELU(),
            torch.nn.Dropout(dropout),
            Linear(hidden // 2, num_classes)
        )
    def forward(self, x, edge_index, batch, edge_attr=None):
        # Ignore graph structure; pool features to graph level first
        graph_feats = global_mean_pool(x, batch)   # [B, 6]
        return self.net(graph_feats)


# ── 2. GCN Baseline ──────────────────────────────────────────────────────────
class GCNClassifier(torch.nn.Module):
    """
    3-layer Graph Convolutional Network (Kipf & Welling, 2017).
    Same depth as ImprovedGAT but NO attention mechanism.
    """
    def __init__(self, num_node_features=6, hidden=64, num_classes=2, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GCNConv(num_node_features, hidden * 4)   # 6 → 256
        self.bn1   = BatchNorm1d(hidden * 4)
        self.conv2 = GCNConv(hidden * 4, hidden * 4)          # 256 → 256
        self.bn2   = BatchNorm1d(hidden * 4)
        self.conv3 = GCNConv(hidden * 4, hidden)              # 256 → 64
        self.bn3   = BatchNorm1d(hidden)
        self.lin1  = Linear(hidden * 2, hidden)               # dual pool
        self.bn4   = BatchNorm1d(hidden)
        self.lin2  = Linear(hidden, hidden // 2)
        self.lin3  = Linear(hidden // 2, num_classes)

    def forward(self, x, edge_index, batch, edge_attr=None):
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn3(self.conv3(x, edge_index)))
        x = torch.cat([global_mean_pool(x, batch),
                        global_max_pool(x, batch)], dim=1)
        x = F.elu(self.bn4(self.lin1(x)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.lin2(x))
        return self.lin3(x)


# ── 3. ImprovedGATClassifier (from NB07) ─────────────────────────────────────
class ImprovedGATClassifier(torch.nn.Module):
    """Copy of the ImprovedGATClassifier from Notebook 07."""
    def __init__(self, num_node_features=6, hidden=64,
                 num_classes=2, heads=4, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(num_node_features, hidden, heads=heads, concat=True, dropout=0.2)
        self.bn1   = BatchNorm1d(hidden * heads)
        self.conv2 = GATConv(hidden * heads, hidden, heads=heads, concat=True, dropout=0.2)
        self.bn2   = BatchNorm1d(hidden * heads)
        self.conv3 = GATConv(hidden * heads, hidden, heads=1, concat=False, dropout=0.2)
        self.bn3   = BatchNorm1d(hidden)
        self.lin1  = Linear(hidden * 2, hidden)
        self.bn4   = BatchNorm1d(hidden)
        self.lin2  = Linear(hidden, hidden // 2)
        self.lin3  = Linear(hidden // 2, num_classes)

    def forward(self, x, edge_index, batch, edge_attr=None):
        x = F.elu(self.bn1(self.conv1(x, edge_index, edge_attr=edge_attr)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn2(self.conv2(x, edge_index, edge_attr=edge_attr)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn3(self.conv3(x, edge_index, edge_attr=edge_attr)))
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        x = F.elu(self.bn4(self.lin1(x)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.lin2(x))
        return self.lin3(x)


print("✅ Model definitions:")
for cls in [MLPClassifier, GCNClassifier, ImprovedGATClassifier]:
    n = sum(p.numel() for p in cls().parameters())
    print(f"   {cls.__name__:<28} {n:>8,} parameters")

✅ Model definitions:
   MLPClassifier                   9,666 parameters
   GCNClassifier                  95,714 parameters
   ImprovedGATClassifier          96,866 parameters


---
## 📦 Section 1: Load Data

In [5]:
# ─── CELL 5: Load Datasets ────────────────────────────────────────────────────
train_dataset = ABIDEGraphDataset(os.path.join(GRAPHS_DIR, 'train'))
val_dataset   = ABIDEGraphDataset(os.path.join(GRAPHS_DIR, 'val'))
test_dataset  = ABIDEGraphDataset(os.path.join(GRAPHS_DIR, 'test'))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

# Compute class weights from training set
train_y  = np.array([d.y.item() for d in train_dataset])
n_train  = len(train_y)
n_asd    = int((train_y == 1).sum())
n_ctrl   = int((train_y == 0).sum())
w_ctrl   = n_train / (2.0 * n_ctrl)
w_asd    = n_train / (2.0 * n_asd)
cw       = torch.tensor([w_ctrl, w_asd], dtype=torch.float)

test_y = np.array([d.y.item() for d in test_dataset])

# Build SVM/MLP feature matrices (flatten connectivity matrix + node features)
print("Building feature matrices for SVM...")

def build_feature_matrix(dataset, mode='mean_features'):
    """
    Flatten graph into a vector for SVM/tabular methods.
    mode='mean_features': mean-pool the 6 node features → [6]
    mode='edge_weights':  use mean+std of edge attrs → [2]
    mode='combined':      both → [8]
    """
    X, y = [], []
    for data in dataset:
        node_mean = data.x.mean(dim=0).numpy()   # [6]
        node_std  = data.x.std(dim=0).numpy()    # [6]
        if data.edge_attr is not None:
            ea_mean = data.edge_attr.mean().item()
            ea_std  = data.edge_attr.std().item()
        else:
            ea_mean, ea_std = 0.0, 0.0
        feats = np.concatenate([node_mean, node_std, [ea_mean, ea_std]])  # [14]
        X.append(feats)
        y.append(data.y.item())
    return np.array(X), np.array(y)

X_train, y_train = build_feature_matrix(train_dataset)
X_val,   y_val   = build_feature_matrix(val_dataset)
X_test,  y_test  = build_feature_matrix(test_dataset)

# Combine train + val for SVM training (more data)
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

print(f"✅ Datasets loaded")
print(f"   Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"   SVM feature vector: {X_train.shape[1]} dimensions")
print(f"   Class weights: ctrl={w_ctrl:.3f}, asd={w_asd:.3f}")

Building feature matrices for SVM...
✅ Datasets loaded
   Train: 609 | Val: 131 | Test: 131
   SVM feature vector: 14 dimensions
   Class weights: ctrl=0.928, asd=1.084


---
## 🔵 Section 2: SVM Baseline

In [6]:
# ─── CELL 6: SVM Classifier ───────────────────────────────────────────────────
print("Training SVM (RBF kernel, class_weight='balanced')...")

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='rbf',
        C=10.0,
        gamma='scale',
        class_weight='balanced',
        probability=True,   # needed for AUC-ROC
        random_state=42
    ))
])

# Train on train+val combined
svm_pipeline.fit(X_trainval, y_trainval)

svm_preds = svm_pipeline.predict(X_test)
svm_probs = svm_pipeline.predict_proba(X_test)[:, 1]

svm_acc  = accuracy_score(y_test, svm_preds)
svm_f1   = f1_score(y_test, svm_preds, average='weighted', zero_division=0)
svm_auc  = roc_auc_score(y_test, svm_probs)
svm_cm   = confusion_matrix(y_test, svm_preds)
tn, fp, fn, tp_s = svm_cm.ravel()
svm_sens = tp_s / (tp_s + fn + 1e-8)
svm_spec = tn  / (tn  + fp + 1e-8)

print(f"\nSVM Results (Test Set, N=131):")
print(f"  Accuracy:    {svm_acc:.4f}")
print(f"  F1:          {svm_f1:.4f}")
print(f"  AUC-ROC:     {svm_auc:.4f}")
print(f"  Sensitivity: {svm_sens:.4f}")
print(f"  Specificity: {svm_spec:.4f}")
print(f"  CM: TP={tp_s} TN={tn} FP={fp} FN={fn}")

Training SVM (RBF kernel, class_weight='balanced')...

SVM Results (Test Set, N=131):
  Accuracy:    0.4809
  F1:          0.4782
  AUC-ROC:     0.4925
  Sensitivity: 0.3934
  Specificity: 0.5571
  CM: TP=24 TN=39 FP=31 FN=37


---
## 🔶 Section 3: MLP & GCN Baselines (Graph Neural Networks)

In [7]:
# ─── CELL 7: Shared GNN Training Function ─────────────────────────────────────

def train_gnn_model(model_class, model_kwargs, train_loader, val_loader,
                     class_weights, device, epochs=200, lr=0.001,
                     model_name='GNN'):
    """
    Generic training loop for any PyG model with the signature
    model(x, edge_index, batch, edge_attr) → logits.
    Returns (trained_model, history_dict).
    """
    model     = model_class(**model_kwargs).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=20, min_lr=1e-5
    )
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))

    best_f1, best_state, best_epoch = 0.0, None, 0
    history = {'val_acc': [], 'val_f1': [], 'val_sens': []}

    print(f"\nTraining {model_name} ({epochs} epochs)...")
    start = time.time()

    for epoch in range(1, epochs + 1):
        # ── Train ────────────────────────────────────────────────────────────
        model.train()
        for data in train_loader:
            data = data.to(device)
            ea   = data.edge_attr.squeeze(-1) if data.edge_attr is not None else None
            optimizer.zero_grad()
            out  = model(data.x, data.edge_index, data.batch, ea)
            loss = criterion(out, data.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # ── Validate ─────────────────────────────────────────────────────────
        model.eval()
        val_preds, val_labels_list = [], []
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                ea   = data.edge_attr.squeeze(-1) if data.edge_attr is not None else None
                out  = model(data.x, data.edge_index, data.batch, ea)
                val_preds.extend(out.argmax(dim=1).cpu().numpy())
                val_labels_list.extend(data.y.cpu().numpy())

        vp  = np.array(val_preds)
        vl  = np.array(val_labels_list)
        vf1 = f1_score(vl, vp, average='weighted', zero_division=0)
        vac = accuracy_score(vl, vp)
        vsen= (vp[vl==1] == 1).mean() if (vl==1).sum() > 0 else 0.0

        scheduler.step(vf1)
        history['val_acc'].append(vac)
        history['val_f1'].append(vf1)
        history['val_sens'].append(float(vsen))

        if vf1 > best_f1:
            best_f1    = vf1
            best_epoch = epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 40 == 0:
            print(f"  Ep {epoch:3d}/{epochs} | val_acc={vac:.4f} | "
                  f"val_f1={vf1:.4f} | best_f1={best_f1:.4f} (ep {best_epoch})",
                  end='\r')

    print(f"\n  Done in {(time.time()-start)/60:.1f} min | "
          f"Best epoch: {best_epoch} | Best val F1: {best_f1:.4f}")

    model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def eval_gnn_test(model, test_loader, device):
    """Evaluate a trained PyG model on the test loader."""
    model.eval()
    preds, labels, probs = [], [], []
    for data in test_loader:
        data = data.to(device)
        ea   = data.edge_attr.squeeze(-1) if data.edge_attr is not None else None
        out  = model(data.x, data.edge_index, data.batch, ea)
        p    = torch.softmax(out, dim=1)
        preds.extend(out.argmax(dim=1).cpu().numpy())
        labels.extend(data.y.cpu().numpy())
        probs.extend(p[:, 1].cpu().numpy())
    preds  = np.array(preds);  labels = np.array(labels);  probs = np.array(probs)
    acc  = accuracy_score(labels, preds)
    f1   = f1_score(labels, preds, average='weighted', zero_division=0)
    auc  = roc_auc_score(labels, probs)
    cm_  = confusion_matrix(labels, preds)
    tn, fp, fn, tp_c = cm_.ravel()
    sens = tp_c / (tp_c + fn + 1e-8)
    spec = tn   / (tn   + fp + 1e-8)
    return acc, f1, auc, float(sens), float(spec), preds, labels, probs


print("✅ Shared training function defined")

✅ Shared training function defined


In [8]:
# ─── CELL 8: Train MLP Baseline ───────────────────────────────────────────────
mlp_model, mlp_history = train_gnn_model(
    MLPClassifier,
    {'in_features': 6, 'hidden': 128, 'num_classes': 2, 'dropout': 0.4},
    train_loader, val_loader, cw, DEVICE, epochs=200, lr=0.001, model_name='MLP'
)

mlp_acc, mlp_f1, mlp_auc, mlp_sens, mlp_spec, mlp_preds, _, mlp_probs = \
    eval_gnn_test(mlp_model, test_loader, DEVICE)

print(f"\nMLP Test Results:")
print(f"  Accuracy={mlp_acc:.4f} | F1={mlp_f1:.4f} | AUC={mlp_auc:.4f} "
      f"| Sens={mlp_sens:.4f} | Spec={mlp_spec:.4f}")


Training MLP (200 epochs)...


ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 128])

In [10]:
# ─── CELL 9: Train GCN Baseline ───────────────────────────────────────────────
gcn_model, gcn_history = train_gnn_model(
    GCNClassifier,
    {'num_node_features': 6, 'hidden': 64, 'num_classes': 2, 'dropout': 0.3},
    train_loader, val_loader, cw, DEVICE, epochs=200, lr=0.001, model_name='GCN'
)

gcn_acc, gcn_f1, gcn_auc, gcn_sens, gcn_spec, gcn_preds, gcn_labels, gcn_probs = \
    eval_gnn_test(gcn_model, test_loader, DEVICE)

print(f"\nGCN Test Results:")
print(f"  Accuracy={gcn_acc:.4f} | F1={gcn_f1:.4f} | AUC={gcn_auc:.4f} "
      f"| Sens={gcn_sens:.4f} | Spec={gcn_spec:.4f}")


Training GCN (200 epochs)...


ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 64])

In [11]:
# ─── CELL 10: Load Improved GAT from Notebook 07 Checkpoint ───────────────────
ckpt_path = os.path.join(IMPROVED_DIR, 'best_improved_gat.pt')

if os.path.exists(ckpt_path):
    gat_model  = ImprovedGATClassifier(
        num_node_features=6, hidden=64, num_classes=2, heads=4, dropout=0.3
    ).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    gat_model.load_state_dict(ckpt['model_state_dict'])
    print(f"✅ Improved GAT loaded from Notebook 07 (Epoch {ckpt['epoch']})")
    print(f"   Val F1 at checkpoint: {ckpt['val_f1']:.4f}")
else:
    print("⚠️  Notebook 07 checkpoint not found — training GAT from scratch...")
    gat_model, _ = train_gnn_model(
        ImprovedGATClassifier,
        {'num_node_features': 6, 'hidden': 64, 'num_classes': 2,
         'heads': 4, 'dropout': 0.3},
        train_loader, val_loader, cw, DEVICE,
        epochs=200, lr=0.001, model_name='ImprovedGAT'
    )

gat_acc, gat_f1, gat_auc, gat_sens, gat_spec, gat_preds, gat_labels, gat_probs = \
    eval_gnn_test(gat_model, test_loader, DEVICE)

print(f"\nImproved GAT Test Results:")
print(f"  Accuracy={gat_acc:.4f} | F1={gat_f1:.4f} | AUC={gat_auc:.4f} "
      f"| Sens={gat_sens:.4f} | Spec={gat_spec:.4f}")

RuntimeError: Error(s) in loading state_dict for ImprovedGATClassifier:
	Missing key(s) in state_dict: "bn4.running_mean", "bn4.running_var". 

---
## 📊 Section 4: Comprehensive Comparison

In [ ]:
# ─── CELL 11: Build Comparison Table ──────────────────────────────────────────

# Load baseline GAT metrics from NB06
old_path = os.path.join(METRICS_DIR, 'evaluation_metrics.csv')
if os.path.exists(old_path):
    old_m = pd.read_csv(old_path).iloc[0]
    base_gat = {'Accuracy': old_m['Accuracy'], 'F1': old_m['F1_Weighted'],
                 'AUC_ROC': old_m['AUC_ROC'], 'Sensitivity': old_m['Sensitivity'],
                 'Specificity': old_m['Specificity']}
else:
    base_gat = {'Accuracy': 0.5267, 'F1': 0.4957,
                 'AUC_ROC': 0.5651, 'Sensitivity': 0.2623, 'Specificity': 0.7571}

comp_data = [
    {'Model': 'SVM (RBF)',
     'Type': 'Classical ML',
     'Graph?': 'No',
     'Attention?': 'No',
     'Accuracy':    round(svm_acc,  4),
     'F1':          round(svm_f1,   4),
     'AUC_ROC':     round(svm_auc,  4),
     'Sensitivity': round(svm_sens, 4),
     'Specificity': round(svm_spec, 4)},
    {'Model': 'MLP',
     'Type': 'Deep Learning',
     'Graph?': 'No',
     'Attention?': 'No',
     'Accuracy':    round(mlp_acc,  4),
     'F1':          round(mlp_f1,   4),
     'AUC_ROC':     round(mlp_auc,  4),
     'Sensitivity': round(mlp_sens, 4),
     'Specificity': round(mlp_spec, 4)},
    {'Model': 'GCN',
     'Type': 'Graph Neural Net',
     'Graph?': 'Yes',
     'Attention?': 'No',
     'Accuracy':    round(gcn_acc,  4),
     'F1':          round(gcn_f1,   4),
     'AUC_ROC':     round(gcn_auc,  4),
     'Sensitivity': round(gcn_sens, 4),
     'Specificity': round(gcn_spec, 4)},
    {'Model': 'Baseline GAT (NB04)',
     'Type': 'Graph Attn Net',
     'Graph?': 'Yes',
     'Attention?': 'Yes',
     'Accuracy':    round(base_gat['Accuracy'],    4),
     'F1':          round(base_gat['F1'],          4),
     'AUC_ROC':     round(base_gat['AUC_ROC'],     4),
     'Sensitivity': round(base_gat['Sensitivity'], 4),
     'Specificity': round(base_gat['Specificity'], 4)},
    {'Model': '★ Improved GAT (Ours)',
     'Type': 'Graph Attn Net',
     'Graph?': 'Yes',
     'Attention?': 'Yes',
     'Accuracy':    round(gat_acc,  4),
     'F1':          round(gat_f1,   4),
     'AUC_ROC':     round(gat_auc,  4),
     'Sensitivity': round(gat_sens, 4),
     'Specificity': round(gat_spec, 4)},
]

comp_df = pd.DataFrame(comp_data)

# Print table
print("=" * 90)
print("  MODEL COMPARISON — ASD Classification on ABIDE I (Test Set, N=131)")
print("=" * 90)
print(f"{'Model':<24} {'Accuracy':>9} {'F1':>8} {'AUC-ROC':>9} "
      f"{'Sensitivity':>12} {'Specificity':>12}")
print("-" * 90)
for row in comp_data:
    marker = '→' if '★' in row['Model'] else ' '
    print(f"{marker} {row['Model']:<22} {row['Accuracy']:>9.4f} {row['F1']:>8.4f} "
          f"{row['AUC_ROC']:>9.4f} {row['Sensitivity']:>12.4f} {row['Specificity']:>12.4f}")
print("=" * 90)

comp_df.to_csv(os.path.join(METRICS_DIR, 'full_model_comparison.csv'), index=False)
print(f"\n✅ Full comparison saved: {METRICS_DIR}/full_model_comparison.csv")

In [ ]:
# ─── CELL 12: Comparison Visualization ────────────────────────────────────────
models_sorted = [r['Model'] for r in comp_data]
short_names   = ['SVM', 'MLP', 'GCN', 'Baseline\nGAT', 'Improved\nGAT (Ours)']
metrics_keys  = ['Accuracy', 'F1', 'AUC_ROC', 'Sensitivity', 'Specificity']
metric_labels = ['Accuracy', 'F1', 'AUC-ROC', 'Sensitivity', 'Specificity']
bar_colors    = ['#9E9E9E', '#78909C', '#546E7A', '#B0BEC5', '#E63946']

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('Model Comparison: SVM vs MLP vs GCN vs GAT\n'
              'ASD Classification — ABIDE I Dataset (Test Set, N=131)',
              fontsize=14, fontweight='bold')

for idx, (metric_key, metric_label) in enumerate(zip(metrics_keys, metric_labels)):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])
    vals = [r[metric_key] for r in comp_data]
    bars = ax.bar(short_names, vals, color=bar_colors, edgecolor='black',
                   linewidth=0.8, alpha=0.9)
    bars[-1].set_edgecolor('gold');  bars[-1].set_linewidth(2.5)  # highlight ours
    ax.axhline(0.5, color='red', lw=1.5, ls='--', alpha=0.6, label='Random (0.5)')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}',
                 ha='center', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 1.12)
    ax.set_ylabel(metric_label)
    ax.set_title(metric_label, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if idx == 0:
        ax.legend(fontsize=8)

# Panel 6: ROC curves for all models
ax6 = fig.add_subplot(gs[1, 2])
roc_models = [
    ('SVM',              y_test, svm_probs, '#9E9E9E', '--'),
    ('MLP',              y_test, mlp_probs, '#546E7A', '--'),
    ('GCN',              gcn_labels, gcn_probs, '#78909C', '--'),
    ('Improved GAT',     gat_labels, gat_probs, '#E63946', '-'),
]
for name, yt, yp, col, ls in roc_models:
    fpr_, tpr_, _ = roc_curve(yt, yp)
    auc_ = roc_auc_score(yt, yp)
    ax6.plot(fpr_, tpr_, color=col, lw=2.0 if col=='#E63946' else 1.5,
              ls=ls, label=f'{name} (AUC={auc_:.3f})')
ax6.plot([0,1],[0,1],'k--', lw=1, alpha=0.5)
ax6.set_xlabel('False Positive Rate')
ax6.set_ylabel('True Positive Rate')
ax6.set_title('ROC Curves — All Models', fontweight='bold')
ax6.legend(loc='lower right', fontsize=8)
ax6.grid(alpha=0.3)
ax6.spines['top'].set_visible(False)
ax6.spines['right'].set_visible(False)

fig_path = os.path.join(FIGURES_DIR, 'full_model_comparison.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {fig_path}")

In [ ]:
# ─── CELL 13: Final Summary ───────────────────────────────────────────────────
best_model = max(comp_data, key=lambda r: r['AUC_ROC'])
print("=" * 70)
print("  NOTEBOOK 08 SUMMARY — Baseline Comparison")
print("=" * 70)
print(f"\n  Best overall model: {best_model['Model']}")
print(f"  Best AUC-ROC:       {best_model['AUC_ROC']:.4f}")
print(f"  Best Sensitivity:   {best_model['Sensitivity']:.4f}")

print("\n📊 Key Findings:")
gat_vs_gcn_acc   = gat_acc  - gcn_acc
gat_vs_svm_sens  = gat_sens - svm_sens
gat_vs_mlp_auc   = gat_auc  - mlp_auc
print(f"  ImprovedGAT vs GCN:    Acc  {gat_vs_gcn_acc:+.4f}  (attention mechanism helps)")
print(f"  ImprovedGAT vs SVM:    Sens {gat_vs_svm_sens:+.4f}  (graph structure helps)")
print(f"  ImprovedGAT vs MLP:    AUC  {gat_vs_mlp_auc:+.4f}  (message passing helps)")

print("\n📁 Files Saved:")
for f in ['results/metrics/full_model_comparison.csv',
           'results/figures/full_model_comparison.png']:
    print(f"  ✅  {f}")

print("\n⏭️  Next Step: Re-run Notebook 06 with the Improved GAT checkpoint")
print("   (see Notebook 07 Cell 14 for exact instructions)")
print("\n" + "=" * 70)